In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from utils import *
import matplotlib.cm as cm

from matplotlib import patheffects as pe

cmap = cm.viridis_r
KAPLAN_COLOR = cmap(0.5)  
DENSE_COLOR = cmap(0.7)      
LSTM_COLOR = cmap(0.9)    

def plot_ceg_growth_rates(axes, C_MIN = 1e19, C_MAX = 1e26, show_legend=False, show_top_xaxis=True, show_bottom_xaxis=True, yaxis_right=False, label_fontsize=12, tick_fontsize=10, include_title=True, year_skip = 2):
    
    assert len(axes) == 3, "Must provide exactly 3 axes"
    
    C = np.logspace(np.log10(C_MIN)-1, np.log10(C_MAX) + 1, 1000)

    mask = (C >= C_MIN) & (C <= C_MAX)
    C = C[mask]

    lstm_to_moe = mixture_of_experts_multiplier(C) * lstm_to_optimal_transformer_multiplier(C, kaplan_divide=False)
    kaplan_to_moe = mixture_of_experts_multiplier(C) * chinchilla_rebalancing_multiplier(C)
    dense_to_moe = mixture_of_experts_multiplier(C)

    years = np.array([cty(c) for c in C])
    lstm_to_moe = np.gradient(np.log(lstm_to_moe), years) * 100
    kaplan_to_moe = np.gradient(np.log(kaplan_to_moe), years) * 100
    dense_to_moe = np.gradient(np.log(dense_to_moe), years) * 100

    min_year = int(np.floor(cty(C_MIN)))
    max_year = int(np.ceil(cty(C_MAX)))


        
    year_range = np.arange(min_year, max_year + 1)
    
    lstm_bars = []
    kaplan_bars = []
    dense_bars = []
    
    for year in year_range:
        compute = ytc(year)
        idx = np.argmin(np.abs(C - compute))
        lstm_bars.append(lstm_to_moe[idx])
        kaplan_bars.append(kaplan_to_moe[idx])
        dense_bars.append(dense_to_moe[idx])
    
    threshold = .1

    for i, (ax, data, color, title) in enumerate([
        (axes[0], lstm_bars, LSTM_COLOR, "Measured Growth Rate Relative to LSTM"),
        (axes[1], kaplan_bars, KAPLAN_COLOR, "Measured Growth Rate Relative to Kaplan Transformer"),
        (axes[2], dense_bars, DENSE_COLOR, "Measured Growth Relative to Dense Transformer")
    ]):
        years_ticks = np.arange(2018, max_year + 1, year_skip)
        tick_labels = [int(year) for year in years_ticks]
        
        ax.set_xticks(years_ticks)
        ax.set_xticklabels(tick_labels)
        ax.minorticks_off()
        
        for year, value in zip(year_range, data):
            if value < threshold:
                ax.bar(year, threshold, color=color, edgecolor=color, linewidth=3, alpha=0.8)
            else:
                ax.bar(year, value, color=color, alpha=0.8)
        
        if include_title:
            ax.set_title(title, fontsize=label_fontsize) #, fontweight='bold')
        ax.set_ylim(-2, 100)
        ax.yaxis.set_major_formatter(PercentFormatter())
        ax.grid(True, alpha=0.5, axis='y')
        
        if i == 0:
            ax.set_ylabel('Annual CEG\nGrowth Rate', fontsize=label_fontsize, fontstyle='italic')
        
        ax.set_xlim(min_year - 0.5, max_year + 0.5)
        
        if i == 1:
            ax.set_xlabel('Model Year', fontsize=label_fontsize, fontstyle='italic')
        
        ax.tick_params(axis='x', which='both', labelsize=tick_fontsize)
        ax.tick_params(axis='y', which='major', labelsize=tick_fontsize)

    plt.tight_layout()


fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharey=True)
plot_ceg_growth_rates(axes)
fig.set_dpi(300)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from utils import *

def plot_ceg_multipliers(axes, normalize=False, C_MIN=None, C_MAX=None, 
                        show_top_xaxis=True, show_bottom_xaxis=True,
                        show_top_xlabel=True, show_bottom_xlabel=True,
                        label_fontsize=12, multiplier_font_size = 30, tick_fontsize=20, year_skip = 2):
    assert len(axes) == 3, "axes must contain exactly 3 axes"
    
    if C_MIN is None:
        C_MIN = ytc(2018)
    if C_MAX is None:
        C_MAX = 1e30

    C = np.logspace(np.log10(C_MIN), np.log10(C_MAX), 1000)
    mask = (C >= C_MIN) & (C <= C_MAX)
    C = C[mask]

    line_configs = [
        ('lstm', mixture_of_experts_multiplier(C) * lstm_to_optimal_transformer_multiplier(C, kaplan_divide=False), LSTM_COLOR, 'Relative to\nLSTM', None),
        ('kaplan', mixture_of_experts_multiplier(C) * chinchilla_rebalancing_multiplier(C), KAPLAN_COLOR, 'Relative to\nKaplan Transformer', ytc(2022)),
        ('dense', mixture_of_experts_multiplier(C), DENSE_COLOR, 'Relative to\nDense Transformer', ytc(2021))
    ]



    for idx, (line_type, values, color, title, threshold) in enumerate(line_configs):
        ax = axes[idx]
        
        if normalize:
            if line_type == 'lstm':
                values = values / lstm_to_optimal_transformer_multiplier(np.array([C_MIN]), kaplan_divide=False)[0]
            elif line_type == 'kaplan':
                values = values / chinchilla_rebalancing_multiplier(np.array([C_MIN]))[0]
            elif line_type == 'dense':
                values = values / mixture_of_experts_multiplier(np.array([C_MIN]))[0]
            else:
                raise Exception(f'Line type {line_type} not recognized.')

        if threshold:
            mask_before = C < threshold
            mask_after = C >= threshold
            ax.plot(C[mask_before], values[mask_before], color=color, linewidth=3, 
                    linestyle='--', alpha=0.8)
            ax.plot(C[mask_after], values[mask_after], color=color, linewidth=3, alpha=0.8)
            ax.fill_between(C[mask_after], values[mask_after], alpha=0.8, color=color)
        else:
            ax.plot(C, values, color=color, linewidth=3, alpha=0.8)
            ax.fill_between(C, values, alpha=0.8, color=color)

        final_value = values[-1]        


        ax.text(C[-1], 5, f'{format_sigfigs(final_value, sigfigs=2)}×', 
                fontsize=multiplier_font_size, fontweight='bold', 
                color='white', ha='right', va='center',
                path_effects=[pe.withStroke(linewidth=2, foreground='black')])

        ax.set_xscale('log')
        ax.set_yscale('log')
        ax.set_xlim(C_MIN, C_MAX)
        
        min_year = np.floor(cty(C_MIN)) + 1
        max_year = np.ceil(cty(C_MAX)) - 1
        years_ticks = np.arange(2018, max_year + 1, year_skip)
        tick_locs = [ytc(year) for year in years_ticks]
        tick_labels = [int(year) for year in years_ticks]
        
        ax.set_xticks(tick_locs)
        ax.set_xticklabels(tick_labels)
        ax.minorticks_off()
        ax.set_yticks([10**i for i in range(1,6) if i % 2 == 1])
        
        if show_bottom_xlabel and idx == 1:
            ax.set_xlabel('Model Year', fontsize=label_fontsize, fontstyle='italic')
        else:
            ax.set_xlabel('')
            
        if show_bottom_xaxis:
            ax.tick_params(axis='x', which='both', labelbottom=True, labelsize=tick_fontsize)
        else:
            ax.tick_params(axis='x', which='both', labelbottom=False)

        if idx == 0:
            if normalize:
                year = round(cty(C_MIN))
                ylabel = f'Compute Equivalent Gains\nIncrease Since {year}'
            else:
                ylabel = 'CEG Multiplier'
            ax.set_ylabel(ylabel, fontsize=label_fontsize, fontstyle='italic', labelpad = 60)

        ax.grid(True, alpha=0.5)
        ax.tick_params(axis='y', which='major', labelsize=tick_fontsize)
        
 
        ax.set_title(title, fontsize=label_fontsize, y=1.4, fontweight='bold')

        ax2 = ax.twiny()
        ax2.set_xscale('log')
        ax2.set_xlim(C_MIN, C_MAX)
        
        if show_top_xlabel and idx == 1:
            ax2.set_xlabel('Compute Frontier (FLOPs)', fontsize=label_fontsize, fontstyle='italic')
            ax2.xaxis.set_label_coords(0.5, 1.25)
        else:
            ax2.set_xlabel('')
            
        if show_top_xaxis:
            ax2.tick_params(axis='x', which='both', labeltop=True, labelsize=tick_fontsize)
        else:
            ax2.tick_params(axis='x', which='both', labeltop=False)

# fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharey=True)

# C_MIN, C_MAX = ytc(2017), ytc(2025)
# LABEL_SIZE, TITLE_SIZE, TICK_SIZE = 13, 17, 10
# plot_ceg_multipliers(list(axes), C_MIN = C_MIN, C_MAX = C_MAX, show_bottom_xaxis=True, label_fontsize = LABEL_SIZE, tick_fontsize=TICK_SIZE, normalize = False, show_bottom_xlabel=False)
# # plt.tight_layout()
# # fig.set_dpi(300)
# # plt.show()

In [ ]:
fig, axs = plt.subplots(nrows = 2, ncols = 3, figsize=(15, 8), sharey='row')
C_MIN, C_MAX = ytc(2017), ytc(2028)

LABEL_SIZE, TITLE_SIZE, TICK_SIZE, YEAR_SKIP = 21, 23, 24,4
plot_ceg_multipliers(axs[0], C_MIN = C_MIN, C_MAX = C_MAX, show_bottom_xaxis=True, label_fontsize = LABEL_SIZE, tick_fontsize=TICK_SIZE, normalize = False, show_bottom_xlabel=False, year_skip=YEAR_SKIP)
plot_ceg_growth_rates(axs[1], C_MIN = C_MIN, C_MAX = C_MAX, show_top_xaxis=False, label_fontsize=LABEL_SIZE, tick_fontsize=TICK_SIZE, yaxis_right=False, include_title=False, year_skip=YEAR_SKIP)

# Add suptitle
fig.suptitle("Growth in Compute Equivalent Gains Depends on the Reference Algorithm", fontsize=TITLE_SIZE, y=.987, fontweight ='bold')

fig.set_dpi(300)
fig.tight_layout(h_pad=0.5)
fig.savefig("figures/CEG_reference_combined.png", dpi = 600, bbox_inches='tight')
plt.show()

In [ ]:
np.log(8460)/np.log(21400)